In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
%pip install -q pandas scikit-learn tqdm

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd
import sklearn
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)
import

pandas: 2.2.2
scikit-learn: 1.6.1


In [ ]:
PROJECT_ROOT = Path(
    "/content/drive/MyDrive/NMA-DL2026/NMA_BrainMRI_Project"
)

DATA_ROOT = PROJECT_ROOT / "data"

EXTRACTED_ROOT = (
    DATA_ROOT
    / "processed"
    / "extracted"
)

RAW_IDS_ROOT = (
    DATA_ROOT
    / "ids"
    / "raw"
)

DATASET_METADATA_ROOT = (
    DATA_ROOT
    / "metadata"
    / "dataset"
)

ORIGINAL_SCRIPT = (
    PROJECT_ROOT
    / "github"
    / "original"
    / "dataset"
    / "code"
    / "02_split_dataset_to_sets.py"
)

RAW_IDS_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

DATASET_METADATA_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

paths = {
    "EXTRACTED_ROOT": EXTRACTED_ROOT,
    "RAW_IDS_ROOT": RAW_IDS_ROOT,
    "DATASET_METADATA_ROOT": DATASET_METADATA_ROOT,
    "ORIGINAL_SCRIPT": ORIGINAL_SCRIPT,
}

for name, path in paths.items():
    print(f"{name:24s}: {path}")
    print(f"{'':24s}  exists = {path.exists()}")

EXTRACTED_ROOT          : /content/drive/MyDrive/NMA-DL2026/NMA_BrainMRI_Project/data/processed/extracted
                          exists = True
RAW_IDS_ROOT            : /content/drive/MyDrive/NMA-DL2026/NMA_BrainMRI_Project/data/ids/raw
                          exists = True
DATASET_METADATA_ROOT   : /content/drive/MyDrive/NMA-DL2026/NMA_BrainMRI_Project/data/metadata/dataset
                          exists = True
ORIGINAL_SCRIPT         : /content/drive/MyDrive/NMA-DL2026/NMA_BrainMRI_Project/github/original/dataset/code/02_split_dataset_to_sets.py
                          exists = True


In [ ]:
assert ORIGINAL_SCRIPT.is_file(), (
    f"Original script was not found:\n{ORIGINAL_SCRIPT}"
)

print(
    ORIGINAL_SCRIPT.read_text(
        encoding="utf-8"
    )
)

"""
Script assigns patients to given set group: train, validation or test.
"""
from json import dumps
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split
from tqdm import tqdm


def create_datalist(set_paths: list[Path], desc: str) -> 'pd.DataFrame':
    data_list = []

    for sub_dir in tqdm(set_paths, desc=desc):
        flair_img_paths = sorted(list(sub_dir.glob('**/*flair*.png')))

        for flair_img_path in flair_img_paths:
            flair_rel_path = flair_img_path.relative_to(sub_dir.parent)
            seg_rel_path = flair_rel_path.parent / (flair_rel_path.name.replace('flair', 'seg'))

            data_list.append({
                'flair': flair_rel_path,
                'seg': seg_rel_path,
                'status': flair_rel_path.name.split('_')[2][:-4]
            })

    return pd.DataFrame(data_list)


def calculate_statistics(data: 'pd.DataFrame') -> dict:
    stats = data_df.groupby('status').size()
    return {
     

In [ ]:
assert EXTRACTED_ROOT.is_dir(), (
    f"Extracted dataset was not found:\n{EXTRACTED_ROOT}"
)

patient_dirs = sorted(
    path
    for path in EXTRACTED_ROOT.iterdir()
    if path.is_dir()
)

print("Patient directories:", len(patient_dirs))

assert len(patient_dirs) == 1251, (
    f"Expected 1,251 patient folders, "
    f"found {len(patient_dirs)}."
)

print("\nFirst ten patient IDs:")
for patient_dir in patient_dirs[:10]:
    print(" ", patient_dir.name)

Patient directories: 1251

First ten patient IDs:
  00000
  00002
  00003
  00005
  00006
  00008
  00009
  00011
  00012
  00014


In [ ]:
incorrect_folders = []

for patient_dir in tqdm(
    patient_dirs,
    desc="Checking patient folders",
):
    flair_files = sorted(
        patient_dir.glob("*_flair_*.png")
    )

    seg_files = sorted(
        patient_dir.glob("*_seg_*.png")
    )

    if (
        len(flair_files) != 6
        or len(seg_files) != 6
    ):
        incorrect_folders.append(
            {
                "patient": patient_dir.name,
                "flair_count": len(flair_files),
                "seg_count": len(seg_files),
            }
        )

print(
    "Incorrect patient folders:",
    len(incorrect_folders),
)

assert not incorrect_folders, (
    incorrect_folders[:10]
)

print(
    "Every patient has "
    "6 FLAIR files and 6 masks."
)

Checking patient folders:   0%|          | 0/1251 [00:00<?, ?it/s]

Incorrect patient folders: 0
Every patient has 6 FLAIR files and 6 masks.


In [ ]:
all_flair_files = list(
    EXTRACTED_ROOT.glob(
        "*/*_flair_*.png"
    )
)

healthy_flair_files = list(
    EXTRACTED_ROOT.glob(
        "*/*_flair_healthy.png"
    )
)

unhealthy_flair_files = list(
    EXTRACTED_ROOT.glob(
        "*/*_flair_unhealthy.png"
    )
)

print("All FLAIR files:", len(all_flair_files))
print(
    "Healthy FLAIR files:",
    len(healthy_flair_files),
)
print(
    "Unhealthy FLAIR files:",
    len(unhealthy_flair_files),
)

assert len(all_flair_files) == 7506
assert (
    len(healthy_flair_files)
    + len(unhealthy_flair_files)
    == 7506
)

overall_healthy_percentage = round(
    len(healthy_flair_files)
    / len(all_flair_files)
    * 100.0,
    2,
)

print(
    "Overall healthy percentage:",
    overall_healthy_percentage,
)

All FLAIR files: 7506
Healthy FLAIR files: 3508
Unhealthy FLAIR files: 3998
Overall healthy percentage: 46.74


In [ ]:
EXTRACTION_METADATA_PATH = (
    DATASET_METADATA_ROOT
    / "01_png_img_extraction.json"
)

assert EXTRACTION_METADATA_PATH.is_file(), (
    "Notebook 01 metadata was not found:\n"
    f"{EXTRACTION_METADATA_PATH}"
)

extraction_metadata = json.loads(
    EXTRACTION_METADATA_PATH.read_text(
        encoding="utf-8"
    )
)

print(
    json.dumps(
        extraction_metadata,
        indent=4,
    )
)

assert extraction_metadata["quantity"] == 7506
assert (
    extraction_metadata["healthy_quantity"]
    == len(healthy_flair_files)
)

print(
    "Notebook 01 metadata matches "
    "the saved PNG filenames."
)

{
    "quantity": 7506,
    "healthy_quantity": 3508,
    "healthy_percentage": 46.74
}
Notebook 01 metadata matches the saved PNG filenames.


In [ ]:
def create_datalist(
    set_paths: list[Path],
    desc: str,
) -> pd.DataFrame:
    data_list: list[dict[str, str]] = []

    for patient_dir in tqdm(
        set_paths,
        desc=desc,
    ):
        flair_img_paths = sorted(
            patient_dir.glob(
                "**/*flair*.png"
            )
        )

        for flair_img_path in flair_img_paths:
            flair_rel_path = (
                flair_img_path.relative_to(
                    patient_dir.parent
                )
            )

            seg_rel_path = (
                flair_rel_path.parent
                / flair_rel_path.name.replace(
                    "flair",
                    "seg",
                )
            )

            status = (
                flair_rel_path.name
                .split("_")[2]
                .removesuffix(".png")
            )

            data_list.append(
                {
                    "flair": (
                        flair_rel_path.as_posix()
                    ),
                    "seg": (
                        seg_rel_path.as_posix()
                    ),
                    "status": status,
                }
            )

    return pd.DataFrame(
        data_list,
        columns=[
            "flair",
            "seg",
            "status",
        ],
    )

In [ ]:
def calculate_statistics(
    data: pd.DataFrame,
) -> dict:
    counts = (
        data
        .groupby("status")
        .size()
    )

    healthy_quantity = int(
        counts.get(
            "healthy",
            0,
        )
    )

    unhealthy_quantity = int(
        counts.get(
            "unhealthy",
            0,
        )
    )

    total = int(
        data.shape[0]
    )

    assert (
        healthy_quantity
        + unhealthy_quantity
        == total
    )

    return {
        "total": total,
        "healthy_quantity": (
            healthy_quantity
        ),
        "healthy_percentage": float(
            round(
                healthy_quantity
                / total
                * 100.0,
                2,
            )
        ),
    }

In [ ]:
sub_dirs = sorted(
    [
        path
        for path in EXTRACTED_ROOT.iterdir()
        if path.is_dir()
    ]
)

whole_dataset_quantity = len(
    sub_dirs
)

train_quantity = int(
    whole_dataset_quantity
    * 0.8
)

val_quantity = (
    whole_dataset_quantity
    - train_quantity
) // 2

train_sub_dirs, test_sub_dirs = (
    train_test_split(
        sub_dirs,
        train_size=train_quantity,
        random_state=0,
    )
)

test_sub_dirs, val_sub_dirs = (
    train_test_split(
        test_sub_dirs,
        train_size=val_quantity,
        random_state=0,
    )
)

print(
    "Total patients:",
    whole_dataset_quantity,
)
print(
    "Train patients:",
    len(train_sub_dirs),
)
print(
    "Validation patients:",
    len(val_sub_dirs),
)
print(
    "Test patients:",
    len(test_sub_dirs),
)

assert len(train_sub_dirs) == 1000
assert len(val_sub_dirs) == 126
assert len(test_sub_dirs) == 125

print(
    "Patient counts match "
    "the repository logic."
)

Total patients: 1251
Train patients: 1000
Validation patients: 126
Test patients: 125
Patient counts match the repository logic.


In [ ]:
train_patient_ids = {
    path.name
    for path in train_sub_dirs
}

validation_patient_ids = {
    path.name
    for path in val_sub_dirs
}

test_patient_ids = {
    path.name
    for path in test_sub_dirs
}

assert train_patient_ids.isdisjoint(
    validation_patient_ids
)

assert train_patient_ids.isdisjoint(
    test_patient_ids
)

assert validation_patient_ids.isdisjoint(
    test_patient_ids
)

all_split_ids = (
    train_patient_ids
    | validation_patient_ids
    | test_patient_ids
)

assert len(all_split_ids) == 1251

print(
    "No patient occurs "
    "in more than one set."
)
print(
    "All 1,251 patients "
    "are represented exactly once."
)

No patient occurs in more than one set.
All 1,251 patients are represented exactly once.


In [ ]:
train_df = create_datalist(
    set_paths=train_sub_dirs,
    desc="Train set",
)

validation_df = create_datalist(
    set_paths=val_sub_dirs,
    desc="Validation set",
)

test_df = create_datalist(
    set_paths=test_sub_dirs,
    desc="Test set",
)

print("Train rows:", len(train_df))
print(
    "Validation rows:",
    len(validation_df),
)
print("Test rows:", len(test_df))

assert len(train_df) == 6000
assert len(validation_df) == 756
assert len(test_df) == 750

print(
    "Image row counts "
    "match the repository split."
)

Train set:   0%|          | 0/1000 [00:00<?, ?it/s]

Validation set:   0%|          | 0/126 [00:00<?, ?it/s]

Test set:   0%|          | 0/125 [00:00<?, ?it/s]

Train rows: 6000
Validation rows: 756
Test rows: 750
Image row counts match the repository split.


In [ ]:
print("Train table:")
display(train_df.head(12))

print("\nValidation table:")
display(validation_df.head(12))

print("\nTest table:")
display(test_df.head(12))

Train table:


,flair,seg,status
0,01480/00_flair_healthy.png,01480/00_seg_healthy.png,healthy
1,01480/01_flair_unhealthy.png,01480/01_seg_unhealthy.png,unhealthy
2,01480/02_flair_unhealthy.png,01480/02_seg_unhealthy.png,unhealthy
3,01480/03_flair_healthy.png,01480/03_seg_healthy.png,healthy
4,01480/04_flair_healthy.png,01480/04_seg_healthy.png,healthy
5,01480/05_flair_healthy.png,01480/05_seg_healthy.png,healthy
6,01590/00_flair_healthy.png,01590/00_seg_healthy.png,healthy
7,01590/01_flair_healthy.png,01590/01_seg_healthy.png,healthy
8,01590/02_flair_unhealthy.png,01590/02_seg_unhealthy.png,unhealthy
9,01590/03_flair_unhealthy.png,01590/03_seg_unhealthy.png,unhealthy



Validation table:


,flair,seg,status
0,00718/00_flair_healthy.png,00718/00_seg_healthy.png,healthy
1,00718/01_flair_unhealthy.png,00718/01_seg_unhealthy.png,unhealthy
2,00718/02_flair_unhealthy.png,00718/02_seg_unhealthy.png,unhealthy
3,00718/03_flair_unhealthy.png,00718/03_seg_unhealthy.png,unhealthy
4,00718/04_flair_unhealthy.png,00718/04_seg_unhealthy.png,unhealthy
5,00718/05_flair_healthy.png,00718/05_seg_healthy.png,healthy
6,01464/00_flair_healthy.png,01464/00_seg_healthy.png,healthy
7,01464/01_flair_healthy.png,01464/01_seg_healthy.png,healthy
8,01464/02_flair_healthy.png,01464/02_seg_healthy.png,healthy
9,01464/03_flair_unhealthy.png,01464/03_seg_unhealthy.png,unhealthy



Test table:


,flair,seg,status
0,01315/00_flair_healthy.png,01315/00_seg_healthy.png,healthy
1,01315/01_flair_healthy.png,01315/01_seg_healthy.png,healthy
2,01315/02_flair_unhealthy.png,01315/02_seg_unhealthy.png,unhealthy
3,01315/03_flair_unhealthy.png,01315/03_seg_unhealthy.png,unhealthy
4,01315/04_flair_unhealthy.png,01315/04_seg_unhealthy.png,unhealthy
5,01315/05_flair_healthy.png,01315/05_seg_healthy.png,healthy
6,01327/00_flair_healthy.png,01327/00_seg_healthy.png,healthy
7,01327/01_flair_healthy.png,01327/01_seg_healthy.png,healthy
8,01327/02_flair_unhealthy.png,01327/02_seg_unhealthy.png,unhealthy
9,01327/03_flair_healthy.png,01327/03_seg_healthy.png,healthy


In [ ]:
def patient_row_counts(
    dataframe: pd.DataFrame,
) -> pd.Series:
    return (
        dataframe["flair"]
        .map(
            lambda value: (
                Path(value).parts[0]
            )
        )
        .value_counts()
    )


for (
    set_name,
    dataframe,
    expected_patient_count,
) in [
    (
        "train",
        train_df,
        1000,
    ),
    (
        "validation",
        validation_df,
        126,
    ),
    (
        "test",
        test_df,
        125,
    ),
]:
    assert list(
        dataframe.columns
    ) == [
        "flair",
        "seg",
        "status",
    ]

    assert not (
        dataframe
        .isna()
        .any()
        .any()
    )

    assert set(
        dataframe["status"].unique()
    ).issubset(
        {
            "healthy",
            "unhealthy",
        }
    )

    assert dataframe["flair"].is_unique
    assert dataframe["seg"].is_unique

    counts = patient_row_counts(
        dataframe
    )

    assert (
        len(counts)
        == expected_patient_count
    )

    assert (
        counts == 6
    ).all()

    print(
        f"{set_name:10s}: "
        f"{expected_patient_count} patients, "
        "6 rows per patient"
    )

train     : 1000 patients, 6 rows per patient
validation: 126 patients, 6 rows per patient
test      : 125 patients, 6 rows per patient


In [ ]:
missing_referenced_paths = []

for set_name, dataframe in {
    "train": train_df,
    "validation": validation_df,
    "test": test_df,
}.items():
    for row in tqdm(
        dataframe.itertuples(
            index=False
        ),
        total=len(dataframe),
        desc=f"Checking {set_name} paths",
    ):
        flair_path = (
            EXTRACTED_ROOT
            / row.flair
        )

        seg_path = (
            EXTRACTED_ROOT
            / row.seg
        )

        if not flair_path.is_file():
            missing_referenced_paths.append(
                str(flair_path)
            )

        if not seg_path.is_file():
            missing_referenced_paths.append(
                str(seg_path)
            )

print(
    "Missing referenced paths:",
    len(missing_referenced_paths),
)

assert not missing_referenced_paths, (
    missing_referenced_paths[:10]
)

print(
    "Every referenced image "
    "and mask exists."
)

Checking train paths:   0%|          | 0/6000 [00:00<?, ?it/s]

Checking validation paths:   0%|          | 0/756 [00:00<?, ?it/s]

Checking test paths:   0%|          | 0/750 [00:00<?, ?it/s]

Missing referenced paths: 0
Every referenced image and mask exists.


In [ ]:
train_stats = calculate_statistics(
    train_df
)

validation_stats = (
    calculate_statistics(
        validation_df
    )
)

test_stats = calculate_statistics(
    test_df
)

metadata = {
    "patients": {
        "quantity": (
            whole_dataset_quantity
        )
    },
    "train": train_stats,
    "validation": validation_stats,
    "test": test_stats,
}

print(
    json.dumps(
        metadata,
        indent=4,
    )
)

split_total_healthy = (
    train_stats["healthy_quantity"]
    + validation_stats[
        "healthy_quantity"
    ]
    + test_stats["healthy_quantity"]
)

assert (
    split_total_healthy
    == len(healthy_flair_files)
)

print(
    "\nTotal healthy slices "
    "across split:",
    split_total_healthy,
)

print(
    "The split preserves "
    "the complete dataset total."
)

{
    "patients": {
        "quantity": 1251
    },
    "train": {
        "total": 6000,
        "healthy_quantity": 2783,
        "healthy_percentage": 46.38
    },
    "validation": {
        "total": 756,
        "healthy_quantity": 360,
        "healthy_percentage": 47.62
    },
    "test": {
        "total": 750,
        "healthy_quantity": 365,
        "healthy_percentage": 48.67
    }
}

Total healthy slices across split: 3508
The split preserves the complete dataset total.


In [ ]:
paper_statistics = {
    "train": {
        "total": 6000,
        "healthy_quantity": 2802,
        "healthy_percentage": 46.70,
    },
    "validation": {
        "total": 756,
        "healthy_quantity": 359,
        "healthy_percentage": 47.49,
    },
    "test": {
        "total": 750,
        "healthy_quantity": 347,
        "healthy_percentage": 46.27,
    },
}

comparison_rows = []

for set_name in [
    "train",
    "validation",
    "test",
]:
    observed = metadata[set_name]
    reported = paper_statistics[
        set_name
    ]

    comparison_rows.append(
        {
            "set": set_name,
            "observed_total": (
                observed["total"]
            ),
            "paper_total": (
                reported["total"]
            ),
            "observed_healthy": (
                observed[
                    "healthy_quantity"
                ]
            ),
            "paper_healthy": (
                reported[
                    "healthy_quantity"
                ]
            ),
            "healthy_difference": (
                observed[
                    "healthy_quantity"
                ]
                - reported[
                    "healthy_quantity"
                ]
            ),
            "observed_percentage": (
                observed[
                    "healthy_percentage"
                ]
            ),
            "paper_percentage": (
                reported[
                    "healthy_percentage"
                ]
            ),
        }
    )

comparison_df = pd.DataFrame(
    comparison_rows
)

display(comparison_df)

if all(
    metadata[set_name]
    == paper_statistics[set_name]
    for set_name in [
        "train",
        "validation",
        "test",
    ]
):
    print(
        "The implementation statistics "
        "match the paper exactly."
    )
else:
    print(
        "The structural split is correct, "
        "but the observed healthy distribution "
        "does not exactly match the paper table."
    )
    print(
        "Do not manually alter the split. "
        "Keep the repository result and "
        "record the paper/code difference."
    )

,set,observed_total,paper_total,observed_healthy,paper_healthy,healthy_difference,observed_percentage,paper_percentage
0,train,6000,6000,2783,2802,-19,46.38,46.70
1,validation,756,756,360,359,1,47.62,47.49
2,test,750,750,365,347,18,48.67,46.27


The structural split is correct, but the observed healthy distribution does not exactly match the paper table.
Do not manually alter the split. Keep the repository result and record the paper/code difference.


In [ ]:
TRAIN_TSV = (
    RAW_IDS_ROOT
    / "train.tsv"
)

VALIDATION_TSV = (
    RAW_IDS_ROOT
    / "validation.tsv"
)

TEST_TSV = (
    RAW_IDS_ROOT
    / "test.tsv"
)

SPLIT_METADATA_PATH = (
    DATASET_METADATA_ROOT
    / "02_split_dataset.json"
)

train_df.to_csv(
    TRAIN_TSV,
    index=False,
    sep="\t",
)

validation_df.to_csv(
    VALIDATION_TSV,
    index=False,
    sep="\t",
)

test_df.to_csv(
    TEST_TSV,
    index=False,
    sep="\t",
)

SPLIT_METADATA_PATH.write_text(
    json.dumps(
        metadata,
        indent=4,
    ),
    encoding="utf-8",
)

print("Saved:")
print(" ", TRAIN_TSV)
print(" ", VALIDATION_TSV)
print(" ", TEST_TSV)
print(" ", SPLIT_METADATA_PATH)

Saved:
  /content/drive/MyDrive/NMA-DL2026/NMA_BrainMRI_Project/data/ids/raw/train.tsv
  /content/drive/MyDrive/NMA-DL2026/NMA_BrainMRI_Project/data/ids/raw/validation.tsv
  /content/drive/MyDrive/NMA-DL2026/NMA_BrainMRI_Project/data/ids/raw/test.tsv
  /content/drive/MyDrive/NMA-DL2026/NMA_BrainMRI_Project/data/metadata/dataset/02_split_dataset.json


In [ ]:
saved_train_df = pd.read_csv(
    TRAIN_TSV,
    sep="\t",
)

saved_validation_df = pd.read_csv(
    VALIDATION_TSV,
    sep="\t",
)

saved_test_df = pd.read_csv(
    TEST_TSV,
    sep="\t",
)

saved_metadata = json.loads(
    SPLIT_METADATA_PATH.read_text(
        encoding="utf-8"
    )
)

assert saved_train_df.equals(
    train_df
)

assert saved_validation_df.equals(
    validation_df
)

assert saved_test_df.equals(
    test_df
)

assert saved_metadata == metadata

print(
    "Saved TSV and JSON files "
    "reopen correctly."
)

Saved TSV and JSON files reopen correctly.


In [ ]:
def patient_ids_from_table(
    dataframe: pd.DataFrame,
) -> set[str]:
    return {
        Path(value).parts[0]
        for value in dataframe["flair"]
    }


saved_train_ids = (
    patient_ids_from_table(
        saved_train_df
    )
)

saved_validation_ids = (
    patient_ids_from_table(
        saved_validation_df
    )
)

saved_test_ids = (
    patient_ids_from_table(
        saved_test_df
    )
)

assert saved_train_ids.isdisjoint(
    saved_validation_ids
)

assert saved_train_ids.isdisjoint(
    saved_test_ids
)

assert saved_validation_ids.isdisjoint(
    saved_test_ids
)

assert len(saved_train_ids) == 1000
assert len(saved_validation_ids) == 126
assert len(saved_test_ids) == 125

print(
    "The saved split "
    "has no patient leakage."
)

The saved split has no patient leakage.
